In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("data/airline.csv")

## Missing value

In [ ]:
# isna() -> True/False, sum() đếm số ô thiếu, mean() * 100 ra tỷ lệ %
missing = pd.DataFrame({
    "so_luong_thieu": df.isna().sum(),
    "ty_le_thieu": df.isna().mean() * 100
})
missing = missing[missing["so_luong_thieu"] > 0]
missing["ty_le_thieu"] = missing["ty_le_thieu"].round(2)
print("Các cột có missing:")
print(missing)
# Lưu ý: không fill Arrival Delay bằng 0, vì 0 là không trễ còn missing là không biết

## Duplicate

In [ ]:
print("Số dòng trùng hoàn toàn:", df.duplicated().sum())
print("Số id bị trùng:", df["id"].duplicated().sum())

## Biến phân loại

In [ ]:
cat_cols = ["Gender", "Customer Type", "Type of Travel", "Class", "satisfaction"]

for col in cat_cols:
    print("\n", col)
    print(df[col].value_counts())

## Biến số

In [ ]:
num_cols = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]
print(df[num_cols].describe().T)

## Outlier của biến số (IQR)

In [ ]:
# Outlier theo quy tắc IQR: nhỏ hơn Q1 - 1.5*IQR hoặc lớn hơn Q3 + 1.5*IQR
# Chỉ in ra để xem, không xóa: delay lớn vẫn có thể là sự kiện thật
for col in num_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df.loc[(df[col] < lower) | (df[col] > upper), col]

    print("\n", col)
    print("Ngưỡng dưới:", lower, "| Ngưỡng trên:", upper)
    print("Số outlier:", len(outliers),
          f"({len(outliers) / df[col].notna().sum() * 100:.2f}%)")
    if len(outliers) > 0:
        print("Các giá trị outlier (giá trị: số lần xuất hiện):")
        print(outliers.value_counts().sort_index())

## Skewness và kurtosis

In [ ]:
# skew: độ lệch, 0 là cân đối, > 0 là lệch phải (đuôi dài bên phải)
# kurt (excess kurtosis): 0 là giống phân phối chuẩn, càng lớn thì đuôi càng dày, nhiều giá trị cực đoan
shape = pd.DataFrame({
    "skewness": df[num_cols].skew(),
    "kurtosis": df[num_cols].kurt()
}).round(2)
print(shape)
# Delay lệch phải rất mạnh -> nên xem median bên cạnh mean, và dùng kiểm định phi tham số (Mann-Whitney U)

## Điểm dịch vụ

In [ ]:
service_cols = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Inflight service",
    "Cleanliness"
]

# Điểm 0 là "not applicable", không phải một mức điểm thật
service_range = pd.DataFrame({
    "min": df[service_cols].min(),
    "max": df[service_cols].max(),
    "so_luong_0": (df[service_cols] == 0).sum()
})
print(service_range)

## Boxplot biến số

In [ ]:
# Outlier không đồng nghĩa dữ liệu sai, delay lớn vẫn có thể là thật nên không xóa
os.makedirs("figures", exist_ok=True)

fig, axes = plt.subplots(1, len(num_cols), figsize=(16, 4))
for ax, col in zip(axes, num_cols):
    ax.boxplot(df[col].dropna())
    ax.set_title(col)
    ax.set_xticks([])

plt.tight_layout()
plt.savefig("figures/02_boxplot_bien_so.png", dpi=150)
plt.show()

## Tương quan giữa các biến số

In [ ]:
# Dùng Spearman vì delay lệch mạnh và nhiều outlier, Pearson dễ bị giá trị cực đoan kéo lệch
num_corr = df[num_cols].corr(method="spearman").round(2)
print(num_corr)

## Tương quan giữa các điểm dịch vụ

In [ ]:
# Bỏ điểm 0 (not applicable) trước khi tính, giống cách xử lý ở file 07
service_data = df[service_cols].mask(df[service_cols] == 0)
service_corr = service_data.corr(method="spearman")

# In các cặp tương quan cao nhất (mỗi cặp chỉ lấy một lần)
pairs = service_corr.where(
    np.triu(np.ones(service_corr.shape, dtype=bool), k=1)
).stack().sort_values(ascending=False)
print("Các cặp tiêu chí tương quan cao nhất:")
print(pairs.head(10).round(2))

# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(service_corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(service_cols)))
ax.set_xticklabels(service_cols, rotation=90)
ax.set_yticks(range(len(service_cols)))
ax.set_yticklabels(service_cols)
for i in range(len(service_cols)):
    for j in range(len(service_cols)):
        ax.text(j, i, f"{service_corr.iloc[i, j]:.2f}",
                ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax)
ax.set_title("Tương quan Spearman giữa các điểm dịch vụ (bỏ điểm 0)")

plt.tight_layout()
plt.savefig("figures/02_heatmap_dich_vu.png", dpi=150)
plt.show()